# Policy Drift OpenEnv — Training Notebook

End-to-end **SFT warm-up + GRPO** fine-tune on the Policy Drift environment.

**Pipeline:**
1. Install deps (Unsloth + TRL 0.24)
2. Clone the repo
3. Generate dataset from the env
4. Load Qwen 2.5 with LoRA adapters
5. SFT warm-up (1 epoch on correct-action pairs)
6. GRPO with 3 independent reward components
7. Offline eval: before / post-SFT / post-GRPO comparison
8. Save LoRA adapters

Set `QUICK_MODE = true` for a ~10 min pipeline validation on Colab T4.  
Set `QUICK_MODE = false` for the onsite run with HF compute credits (A100/H100).

## 1. Install dependencies

Unsloth needs `trl>=0.18.2`. Do NOT pin older TRL or the install breaks.

In [ ]:
!pip install --upgrade -q pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U "trl==0.24.0" datasets accelerate peft bitsandbytes
# IMPORTANT: restart runtime AFTER this cell finishes, BEFORE running anything below.
# (Runtime -> Restart runtime)

## 2. Clone the repo (idempotent)

Run this AFTER restarting the runtime. Safe to re-run — won't re-clone.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/shreyas-garg/OpenEnv.git'
REPO_DIR = '/content/OpenEnv'

if not os.path.isdir(os.path.join(REPO_DIR, 'drift_env')):
    # Only clone if not already cloned (idempotent, avoids nested clones)
    if os.path.isdir(REPO_DIR):
        !rm -rf {REPO_DIR}
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Repo exists — just pull latest
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('cwd:', os.getcwd())
print('drift_env present:', os.path.isdir('drift_env'))
print('train.py present:', os.path.isfile('train.py'))

## 3. Pick run mode

`QUICK_MODE=true` → Qwen 2.5 0.5B, 50 episodes, 50 GRPO steps. ~10 min on Colab T4.  
`QUICK_MODE=false` → Qwen 2.5 3B, 800 episodes, 600 GRPO steps. Onsite only.

In [ ]:
os.environ['QUICK_MODE'] = 'true'
os.environ['USE_WANDB'] = 'false'  # set to 'true' + set WANDB_API_KEY for live logging

## 4. Sanity check — env + dataset work

In [ ]:
from drift_env.dataset import build_dataset, dataset_stats
rows = build_dataset(n_episodes=10, start_seed=0)
print(dataset_stats(rows))
print('\nSample prompt (first 400 chars):\n' + rows[5]['prompt'][:400])

## 5. Run the full pipeline

Output will show: pre-training eval → SFT → post-SFT eval → GRPO → post-GRPO eval.

**Key number to watch: `drift-sens acc`** at each eval stage. Should increase.

In [ ]:
!python train.py

## 6. Inspect saved LoRA adapters

After the run, adapters live in `./outputs/lora_adapters/`. Do NOT naively merge 4-bit bases → 16-bit + adapters (Unsloth warning). Use the dedicated merge path if you need a single checkpoint.

In [ ]:
!ls -la outputs/lora_adapters/ 2>/dev/null || echo 'train.py did not finish — check the run output above'

## Scale-up notes for onsite

- Set `os.environ['QUICK_MODE'] = 'false'`.
- Default non-quick model is `unsloth/Qwen2.5-3B-Instruct`. For bigger GPUs, override: `os.environ['MODEL_NAME'] = 'unsloth/Qwen2.5-7B-Instruct'`.
- T4 → fp16, A100/H100 → bf16 (auto-detected in `train.py`).
- Enable wandb: `os.environ['WANDB_API_KEY']='...'`, then `os.environ['USE_WANDB']='true'`.
- Checkpoint every 100 steps once the curve is moving — in case you need to roll back.